In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from ocr_vs_vlm.results_final.shared.colors import (
    get_model_color, get_dataset_color, APPROACH_COLORS, AZURE, OPENAI, CLAUDE
)
from ocr_vs_vlm.results_final.shared.stats_utils import (
    bootstrap_ci, wilcoxon_test, cohens_d, effect_size_interpretation, bonferroni_correction
)
from ocr_vs_vlm.results_final.shared.viz_utils import (
    setup_plotly_template, create_metric_boxplot, create_radar_chart, create_pareto_chart
)
from ocr_vs_vlm.results_final.shared.data_loader import (
    load_all_data, extract_models_from_columns, PHASE_TO_APPROACH, load_pricing
)

setup_plotly_template()
print("✓ Setup complete")

✓ Setup complete


## 1. Load All QA Data

In [2]:
QA_DATASETS = ['DocVQA_mini', 'InfographicVQA_mini']

try:
    df = load_all_data(datasets=QA_DATASETS, task_type='qa')
    print(f"✓ Loaded {len(df)} total samples")
    print(f"  Datasets: {df['dataset'].unique().tolist()}")
    print(f"  Approaches: {df['approach'].unique().tolist()}")
except Exception as e:
    print(f"Error: {e}")
    df = pd.DataFrame()

models = extract_models_from_columns(df) if len(df) > 0 else []
print(f"  Models: {models}")

✓ Loaded 9520 total samples
  Datasets: ['DocVQA_mini', 'InfographicVQA_mini']
  Approaches: ['unknown']
  Models: ['azure_intelligence', 'claude_sonnet', 'gpt-5-mini', 'gpt-5-nano', 'mistral_document_ai', 'pre_extracted_ocr']


## 2. Approach Comparison (All Datasets)

In [3]:
# Aggregate ANLS by approach
approach_stats = []

for approach in df['approach'].unique() if len(df) > 0 else []:
    app_df = df[df['approach'] == approach]
    
    all_scores = []
    for model in models:
        col = f'anls_score_{model}'
        if col in app_df.columns:
            scores = app_df[col].dropna().values
            all_scores.extend(scores)
            
            if len(scores) > 0:
                mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                approach_stats.append({
                    'Approach': approach,
                    'Model': model,
                    'ANLS': mean,
                    'CI_Lower': ci_lo,
                    'CI_Upper': ci_hi,
                    'N': len(scores)
                })

stats_df = pd.DataFrame(approach_stats)
if len(stats_df) > 0:
    display(Markdown("### ANLS by Approach and Model"))
    display(stats_df.pivot_table(index='Model', columns='Approach', values='ANLS').round(4))

### ANLS by Approach and Model

Approach,unknown
Model,
azure_intelligence,0.6151
claude_sonnet,0.0128
gpt-5-mini,0.5781
gpt-5-nano,0.3905
mistral_document_ai,0.4637
pre_extracted_ocr,0.3936


In [4]:
# Approach summary
if len(stats_df) > 0:
    approach_summary = stats_df.groupby('Approach').agg({
        'ANLS': ['mean', 'std', 'count']
    }).round(4)
    approach_summary.columns = ['Mean ANLS', 'Std', 'N Models']
    approach_summary = approach_summary.sort_values('Mean ANLS', ascending=False)
    
    display(Markdown("### Approach Summary"))
    display(approach_summary)

### Approach Summary

,Mean ANLS,Std,N Models
Approach,,,
unknown,0.409,0.2152,6


In [5]:
# Visualization: Box plot by approach
if len(stats_df) > 0:
    fig = go.Figure()
    
    approach_colors = {
        'ocr_pipeline': AZURE['primary'],
        'vlm_pipeline': OPENAI['primary'],
        'direct_vqa': CLAUDE['primary'],
        'preextracted': '#6B7280'
    }
    
    for approach in stats_df['Approach'].unique():
        app_data = stats_df[stats_df['Approach'] == approach]
        color = approach_colors.get(approach, '#6B7280')
        
        fig.add_trace(go.Box(
            y=app_data['ANLS'],
            name=approach.replace('_', ' ').title(),
            marker_color=color,
            boxpoints='all',
            jitter=0.3
        ))
    
    fig.update_layout(
        title='QA Approaches: ANLS Distribution (Higher is Better)',
        yaxis_title='ANLS Score',
        height=500,
        showlegend=False
    )
    fig.show()

## 3. Statistical Significance: Approach Comparison

In [6]:
# Pairwise approach comparison
print("📊 Pairwise Approach Comparison (Wilcoxon)")
print("=" * 70)

approaches = df['approach'].unique().tolist() if len(df) > 0 else []
approach_scores = {}

# Collect all ANLS scores per approach
for approach in approaches:
    app_df = df[df['approach'] == approach]
    all_scores = []
    for model in models:
        col = f'anls_score_{model}'
        if col in app_df.columns:
            all_scores.extend(app_df[col].dropna().tolist())
    approach_scores[approach] = np.array(all_scores)

# Pairwise tests
p_values = []
comparisons = []

for i, a1 in enumerate(approaches):
    for a2 in approaches[i+1:]:
        if len(approach_scores[a1]) > 0 and len(approach_scores[a2]) > 0:
            n = min(len(approach_scores[a1]), len(approach_scores[a2]))
            stat, p = wilcoxon_test(approach_scores[a1][:n], approach_scores[a2][:n])
            d = cohens_d(approach_scores[a1], approach_scores[a2])
            
            p_values.append(p)
            comparisons.append((a1, a2, p, d))

# Bonferroni correction
significant = bonferroni_correction(p_values) if p_values else []

for (a1, a2, p, d), sig in zip(comparisons, significant):
    m1, m2 = np.mean(approach_scores[a1]), np.mean(approach_scores[a2])
    winner = a1 if m1 > m2 else a2
    sig_mark = '✓' if sig else '✗'
    
    print(f"\n{a1} vs {a2}:")
    print(f"   {a1}: {m1:.4f} | {a2}: {m2:.4f}")
    print(f"   Winner: {winner} | p={p:.4f} | d={d:.3f} ({effect_size_interpretation(d)})")
    print(f"   {sig_mark} Significant after Bonferroni correction")

📊 Pairwise Approach Comparison (Wilcoxon)


## 4. Dataset Breakdown

In [7]:
# Approach performance by dataset
if len(df) > 0:
    fig = make_subplots(rows=1, cols=2, subplot_titles=QA_DATASETS)
    
    for i, dataset in enumerate(QA_DATASETS):
        ds_df = df[df['dataset'] == dataset]
        
        for approach in ds_df['approach'].unique():
            app_df = ds_df[ds_df['approach'] == approach]
            
            all_scores = []
            for model in models:
                col = f'anls_score_{model}'
                if col in app_df.columns:
                    all_scores.extend(app_df[col].dropna().tolist())
            
            if all_scores:
                color = APPROACH_COLORS.get(approach, '#6B7280')
                fig.add_trace(
                    go.Box(y=all_scores, name=approach, marker_color=color,
                          showlegend=(i == 0)),
                    row=1, col=i+1
                )
    
    fig.update_layout(
        title='QA Approaches by Dataset',
        height=450,
        boxmode='group'
    )
    fig.show()

## 5. Cost-Performance Analysis

In [8]:
# Estimate cost per approach
pricing = load_pricing()

# Cost model (rough estimates per sample)
approach_costs = {
    'ocr_pipeline': 0.002,  # OCR + LLM call
    'vlm_pipeline': 0.008,  # VLM parse + LLM call (2 API calls)
    'direct_vqa': 0.004,    # Single VLM call with image
    'preextracted': 0.001   # Only LLM call (OCR pre-done)
}

if len(stats_df) > 0:
    # Create cost-performance data
    approach_perf = stats_df.groupby('Approach')['ANLS'].mean().reset_index()
    approach_perf['Cost'] = approach_perf['Approach'].map(approach_costs)
    approach_perf = approach_perf.dropna()
    
    fig = create_pareto_chart(
        approach_perf,
        x_col='Cost',
        y_col='ANLS',
        label_col='Approach',
        title='QA: Cost vs Performance by Approach',
        x_label='Estimated Cost per Sample ($)',
        y_label='ANLS Score'
    )
    fig.show()

KeyError: 'Cost'

## 6. Key Conclusions

In [ ]:
print("=" * 70)
print("GLOBAL QA COMPARISON: KEY CONCLUSIONS")
print("=" * 70)

if len(approach_scores) > 0:
    print("\n📊 Approach Ranking (by mean ANLS):")
    ranked = sorted([(a, np.mean(s)) for a, s in approach_scores.items() if len(s) > 0], 
                   key=lambda x: -x[1])
    for i, (approach, mean) in enumerate(ranked, 1):
        print(f"   {i}. {approach}: {mean:.4f}")
    
    best_approach = ranked[0][0]
    print(f"\n🏆 Best Approach: {best_approach}")
    
    print("\n💡 Recommendations:")
    if 'ocr_pipeline' in [r[0] for r in ranked[:2]]:
        print("   → OCR Pipeline is highly competitive for document QA")
        print("   → Best cost-efficiency when OCR is already available")
    if 'direct_vqa' in [r[0] for r in ranked[:2]]:
        print("   → Direct VQA offers strong performance with simpler pipeline")
        print("   → Consider for complex visual-textual questions")

print("\n" + "=" * 70)